# Titanic Survival Prediction — First Machine Learning Classifier

**Objective:**
Train a **Decision Tree Classifier** using Scikit-Learn to predict survival on the Titanic, evaluate it strictly on unseen test data, and compare its performance against a simple majority-class baseline.

### Core Requirements Addressed:
1. **Train / Test Split** with fixed `random_state` for reproducible evaluation.
2. **Model Training** with Scikit-Learn (`DecisionTreeClassifier` and `DummyClassifier`).
3. **Evaluation Metrics**: Accuracy, Confusion Matrix, and Mean Squared Error (MSE).
4. **Baseline Comparison**: Outperforming the trivial 'everyone perishes' baseline.
5. **Overfitting Analysis**: Understanding train vs. test generalization and tree depth regularization.

## 1. Setup and Library Imports
We import standard scientific Python packages and Scikit-Learn tools.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    mean_squared_error
)

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 120

## 2. Load the Titanic Dataset
We load the Titanic dataset from the local file if present, or download directly from GitHub.

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
try:
    df = pd.read_csv("titanic.csv" if os.path.exists("titanic.csv") else url)
except Exception:
    df = sns.load_dataset("titanic")
    if "survived" in df.columns:
        df = df.rename(columns={"survived": "Survived", "pclass": "Pclass", "sex": "Sex", "age": "Age", "fare": "Fare"})

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### Target Variable Distribution
Let's inspect the balance between survivors (1) and victims (0). In the Titanic disaster, approximately 61.6% of passengers perished.

In [ ]:
class_counts = df['Survived'].value_counts()
print("Class Distribution:")
print(f"  Perished (0): {class_counts[0]} ({class_counts[0]/len(df)*100:.2f}%)")
print(f"  Survived (1): {class_counts[1]} ({class_counts[1]/len(df)*100:.2f}%)")

## 3. Data Cleaning and Preprocessing
Machine learning models require numerical inputs without missing values:
- Impute missing `Age` using the median.
- Impute missing `Fare` using the median.
- Encode `Sex`: `female` -> 1, `male` -> 0.
- One-hot encode categorical port of embarkation (`Embarked`).

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

cols_to_use = [col for col in features if col in df.columns]
data = df[cols_to_use + [target]].copy()

# Imputation
data['Age'] = data['Age'].fillna(data['Age'].median())
if 'Fare' in data.columns:
    data['Fare'] = data['Fare'].fillna(data['Fare'].median())
if 'Embarked' in data.columns:
    data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])

# Binary Encoding
data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})

# Categorical One-Hot Encoding
if 'Embarked' in data.columns:
    data = pd.get_dummies(data, columns=['Embarked'], drop_first=True)

X = data.drop(columns=[target])
y = data[target].astype(int)

print("Cleaned feature matrix preview:")
X.head()

## 4. Train / Test Split
> **Golden Rule:** *Never evaluate a model on the data it trained on.*

We use `train_test_split` with `test_size=0.20` and `random_state=42` to guarantee consistent, reproducible results. `stratify=y` preserves the class ratio in both splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set samples: {X_train.shape[0]}")
print(f"Testing set samples : {X_test.shape[0]} (unseen holdout data)")

## 5. Simple Baseline Model
A **Dummy Classifier** that always predicts the majority class (`0` = Perished) serves as our reference baseline. Any intelligent model must demonstrate clear predictive improvement over this baseline.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

y_baseline_pred = baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, y_baseline_pred)
baseline_mse = mean_squared_error(y_test, y_baseline_pred)

print(f"Baseline Test Accuracy : {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")
print(f"Baseline Test MSE      : {baseline_mse:.4f}")

## 6. Training the Decision Tree Classifier

### Understanding Overfitting
Decision trees without depth constraints can split until every training leaf is pure (100% training accuracy), but they generalize poorly to new data. Let's compare an **unconstrained tree** vs. a **pruned tree** (`max_depth=3`).

In [ ]:
# Overfitting demonstration
overfitted_tree = DecisionTreeClassifier(random_state=42)
overfitted_tree.fit(X_train, y_train)
print("--- Overfitted Tree (max_depth=None) ---")
print(f"Train Accuracy: {accuracy_score(y_train, overfitted_tree.predict(X_train))*100:.2f}%")
print(f"Test Accuracy : {accuracy_score(y_test, overfitted_tree.predict(X_test))*100:.2f}%")

# Regularized Decision Tree
model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42)
model.fit(X_train, y_train)
print("\n--- Regularized Tree (max_depth=3) ---")
print(f"Train Accuracy: {accuracy_score(y_train, model.predict(X_train))*100:.2f}%")
print(f"Test Accuracy : {accuracy_score(y_test, model.predict(X_test))*100:.2f}%")

## 7. Model Evaluation on Unseen Test Data
We evaluate the pruned Decision Tree using:
1. **Accuracy**
2. **Mean Squared Error (MSE)**
3. **Confusion Matrix & Classification Report**

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Metrics
acc = accuracy_score(y_test, y_pred)
mse_discrete = mean_squared_error(y_test, y_pred)
mse_prob = mean_squared_error(y_test, y_prob)

print(f"Accuracy Score                     : {acc:.4f} ({acc*100:.2f}%)")
print(f"Mean Squared Error (Discrete Labels): {mse_discrete:.4f}")
print(f"Mean Squared Error (Probabilities)  : {mse_prob:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Perished (0)', 'Survived (1)']))

### Confusion Matrix Breakdown

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Perished (0)', 'Survived (1)'])
disp.plot(cmap='Blues', values_format='d', ax=ax)
ax.set_title("Decision Tree Confusion Matrix (Test Data)", fontsize=13, fontweight='bold')
plt.show()

### Model Comparison Summary

In [ ]:
results_df = pd.DataFrame({
    'Model': ['Dummy Baseline (Majority Class)', 'Decision Tree (max_depth=3)'],
    'Test Accuracy': [f"{baseline_acc*100:.2f}%", f"{acc*100:.2f}%"],
    'Test MSE': [f"{baseline_mse:.4f}", f"{mse_discrete:.4f}"],
    'Accuracy Gain': ['0.00%', f"+{(acc - baseline_acc)*100:.2f}%"]
})
results_df

## 8. Feature Importance and Tree Visualization

In [ ]:
# Visualizing the tree logic
plt.figure(figsize=(16, 9))
plot_tree(
    model,
    feature_names=X.columns,
    class_names=['Perished', 'Survived'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Visualizing the Learned Decision Tree (max_depth=3)", fontsize=14, fontweight='bold')
plt.show()

# Feature Importance Bar Chart
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title("Feature Importance in Survival Prediction", fontsize=13, fontweight='bold')
plt.xlabel("Importance Score")
plt.show()